# AP FRQ Auto-Grader — batch by subject folder

Grade handwritten AP free-response answers against the official marking scheme, **one subject folder at a time**.

**Pipeline (per folder):** question PDF + answer PDF + marking-scheme PDF → Gemini handwriting OCR → marking-scheme parse (cached) → per-question grading (unattempted sub-parts scored 0/max) → self-contained HTML scorecard.

Drop a full set of PDFs into `data/<subject-slug>/` (e.g. `data/calculus-bc/`) and **Run All**. Every folder that has all three PDFs is graded, and a report is written to `out/<slug>_ai_scorecard.html`.

## Setup

1. `pip install -r requirements.txt` (or `uv pip install -r requirements.txt`).
2. Copy `.env.example` → `.env` and pick ONE auth method:
   - **AI Studio API key (recommended):** paste your key into `GEMINI_API_KEY=`. Get one at https://aistudio.google.com/apikey.
   - **Vertex AI service account:** set `GOOGLE_APPLICATION_CREDENTIALS` + `GOOGLE_CLOUD_PROJECT`.
3. Put each exam's three PDFs into its subject folder under `data/`:
   - `data/<slug>/questions.pdf` — the typed exam questions
   - `data/<slug>/answers.pdf` — the student's scanned handwritten responses
   - `data/<slug>/marking-scheme.pdf` — the typed rubric / model answers

   Use the folder slugs registered in `config.py` (e.g. `calculus-bc`, `psychology`, `human-geography`). Filenames are matched loosely, so `marking scheme.pdf` works too.
4. Run all cells.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

from dotenv import load_dotenv

from helpers import (
    get_gemini_client,
    discover_exam_folders,
    grade_exam,
    render_html_report,
    render_pdf_to_images,
)
from schemas import ParsedSubmission, Scorecard
from config import SUBJECT_SLUG, SUBJECT_GRADING_ADDENDA, SUBJECT_OCR_ADDENDA

# override=True so edits to .env (e.g. GOOGLE_CLOUD_LOCATION) take effect on a
# re-run without needing a full kernel restart.
load_dotenv(override=True)  # reads Grader/.env

# Reverse map: folder slug ("calculus-bc") -> canonical subject ("AP Calculus BC").
SLUG_TO_SUBJECT = {slug: subject for subject, slug in SUBJECT_SLUG.items()}

In [ ]:
CONFIG = {
    # Batch input: every data/<subject-slug>/ folder that holds a full set of
    # PDFs (questions + answers + marking scheme) is discovered and graded.
    "data_dir":             Path("data/"),
    "only_subjects":        None,           # e.g. ["calculus-bc"] to grade one; None = every folder found

    # Skip any folder whose scorecard already exists in `output_dir` (both the
    # .html and .json must be present). Flip to True to re-grade everything, or
    # delete a single out/<slug>_ai_scorecard.{html,json,md} set to re-run one.
    "force_regrade":        False,

    # Exam metadata applied to every folder (the folder name picks the subject).
    "year":                 2024,
    "set":                  None,           # "Set 1" / "Set 2" / None
    "questions":            "all",          # or a list of sub-part ids: ["1a", "3b"]

    # Models (Gemini 3.x). Valid IDs on Vertex: "gemini-3.5-flash" (GA, fast) and
    # "gemini-3.1-pro-preview" (the Pro tier — stronger vision, slower, preview).
    # NOTE: there is no "gemini-3.5-pro".
    "model_ocr":            "gemini-3.1-pro-preview",  # handwriting OCR → use Pro for accuracy
    "model_rubric":         "gemini-3.5-flash",        # typed marking scheme (cached) → Flash is plenty
    "model_grading":        "gemini-3.5-flash",
    "ocr_dpi":              300,
    "rubric_dpi":           200,            # typed text needs less than handwriting
    "ocr_thinking_level":   "low",          # "minimal" / "low" / "medium" / "high"

    # Grading concurrency — questions are graded in parallel (I/O-bound API calls).
    "grading_max_workers":  8,

    # Output: one self-contained report per folder, named <slug>_ai_scorecard.html
    "output_dir":           Path("out/"),

    # Confidence handling (no human checkpoint, so low confidence is surfaced
    # in the scorecard rather than blocking).
    "low_confidence_threshold": 0.75,
}
print(f"{len(SUBJECT_SLUG)} subjects supported. Scanning: {CONFIG['data_dir'].resolve()}")

## Discover exam folders

Scan `data/` for subject folders that contain a full set of PDFs (questions + answers + marking scheme). Empty or incomplete folders are listed and skipped — nothing fails silently.

In [ ]:
exams, notes = discover_exam_folders(
    CONFIG["data_dir"],
    SLUG_TO_SUBJECT,
    only_slugs=CONFIG["only_subjects"],
)

for n in notes:
    print("•", n)

if not exams:
    raise FileNotFoundError(
        f"No gradable folders found under {CONFIG['data_dir'].resolve()}.\n"
        "Put questions.pdf, answers.pdf and marking-scheme.pdf inside a subject\n"
        "folder, e.g. data/calculus-bc/. Known slugs:\n  - "
        + "\n  - ".join(sorted(SLUG_TO_SUBJECT))
    )

print(f"\nWill grade {len(exams)} exam folder(s):")
for e in exams:
    print(f"  - {e['slug']:32s} → {e['subject']}")

## Grade each folder

For every discovered folder: run OCR, parse the marking scheme (cached per folder), grade each sub-part, and write `out/<slug>_ai_scorecard.{html,json,md}`. Unattempted sub-parts are scored 0/max. A folder that errors is reported and skipped without aborting the rest.

In [ ]:
import time

client = get_gemini_client()
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)

if not CONFIG["force_regrade"]:
    print("Caching ON: folders whose scorecard already exists in out/ skip the "
          "expensive Gemini grading pass. The HTML is always re-rendered from "
          "the cached bundle on every run, so renderer code changes take effect "
          "without re-grading. Caches missing the .bundle.json sidecar (from "
          "pre-bundle runs) are automatically re-graded once to upgrade. Set "
          "CONFIG['force_regrade']=True to force a full re-grade.")

results = []   # (slug, subject, scorecard, html_path)
failures = []  # (slug, subject, error)
cached = []    # slugs whose scorecard was loaded from disk instead of regraded

for e in exams:
    slug, subject = e["slug"], e["subject"]
    print(f"\n=== {subject}  ({slug}) ===")

    base = CONFIG["output_dir"] / f"{slug}_ai_scorecard"
    html_path = base.with_suffix(".html")
    json_path = base.with_suffix(".json")
    # Sidecar with everything render_html_report needs to re-run without
    # calling Gemini (submission + recovered_qids + merged_parent_answers).
    # A cache without this file cannot be re-rendered, so we treat it as a
    # miss and re-grade.
    bundle_path = base.with_suffix(".bundle.json")

    cache_complete = (
        not CONFIG["force_regrade"]
        and html_path.exists()
        and json_path.exists()
        and bundle_path.exists()
    )
    if cache_complete:
        # Re-render HTML from cached data so renderer updates take effect.
        from schemas import TranscribedAnswer
        sc = Scorecard.model_validate_json(json_path.read_text(encoding="utf-8"))
        bundle = json.loads(bundle_path.read_text(encoding="utf-8"))
        submission = ParsedSubmission.model_validate(bundle["submission"])
        recovered_qids = bundle.get("recovered_qids", [])
        merged_parent_answers = {
            qid: TranscribedAnswer.model_validate(d)
            for qid, d in bundle.get("merged_parent_answers", {}).items()
        }
        answer_images = render_pdf_to_images(e["answers_pdf"], dpi=CONFIG["ocr_dpi"])
        html_report = render_html_report(
            scorecard=sc, submission=submission, answer_images=answer_images,
            low_confidence_threshold=CONFIG["low_confidence_threshold"],
            recovered_qids=recovered_qids,
            merged_parent_answers=merged_parent_answers,
        )
        html_path.write_text(html_report, encoding="utf-8")
        results.append((slug, subject, sc, html_path))
        cached.append(slug)
        print(f"  cached: {sc.total_points_earned:.1f}/{sc.total_points_possible:.1f} "
              f"({sc.percentage:.1f}%)  ->  {html_path.name}  (HTML re-rendered from bundle)")
        continue

    # Cache miss OR incomplete (no bundle yet) — do a fresh grade and write
    # all three artefacts so future runs hit the fast re-render path.
    if not CONFIG["force_regrade"] and html_path.exists() and not bundle_path.exists():
        print(f"  upgrading legacy cache: no .bundle.json — re-grading once to populate it")

    _t0 = time.perf_counter()
    try:
        out = grade_exam(
            client,
            subject=subject,
            year=CONFIG["year"],
            set_label=CONFIG["set"],
            questions_pdf=e["questions_pdf"],
            answers_pdf=e["answers_pdf"],
            marking_scheme_pdf=e["marking_scheme_pdf"],
            ocr_prompt_path=Path("prompts/ocr.txt"),
            rubric_prompt_path=Path("prompts/rubric_extract.txt"),
            grade_prompt_path=Path("prompts/grade_question.txt"),
            subject_addendum=SUBJECT_GRADING_ADDENDA.get(subject, ""),
            ocr_subject_addendum=SUBJECT_OCR_ADDENDA.get(subject, ""),
            model_ocr=CONFIG["model_ocr"],
            model_rubric=CONFIG["model_rubric"],
            model_grading=CONFIG["model_grading"],
            ocr_dpi=CONFIG["ocr_dpi"],
            rubric_dpi=CONFIG["rubric_dpi"],
            ocr_thinking_level=CONFIG["ocr_thinking_level"],
            grading_max_workers=CONFIG["grading_max_workers"],
            low_confidence_threshold=CONFIG["low_confidence_threshold"],
            questions=CONFIG["questions"],
            config_echo={
                "slug": slug,
                "models": [CONFIG["model_ocr"], CONFIG["model_rubric"], CONFIG["model_grading"]],
                "year": CONFIG["year"],
                "set": CONFIG["set"],
            },
        )
    except Exception as exc:
        print(f"  ✗ FAILED: {exc}")
        failures.append((slug, subject, str(exc)))
        continue

    scorecard = out["scorecard"]
    merged_parent_answers = out.get("merged_parent_answers", {})
    html_report = render_html_report(
        scorecard=scorecard,
        submission=out["submission"],
        answer_images=out["answer_images"],
        low_confidence_threshold=CONFIG["low_confidence_threshold"],
        recovered_qids=out.get("recovered_qids", []),
        merged_parent_answers=merged_parent_answers,
    )

    html_path.write_text(html_report, encoding="utf-8")
    json_path.write_text(scorecard.model_dump_json(indent=2), encoding="utf-8")
    # Save the render bundle so later runs can re-render HTML without Gemini.
    bundle_path.write_text(
        json.dumps(
            {
                "submission": out["submission"].model_dump(),
                "recovered_qids": out.get("recovered_qids", []),
                "merged_parent_answers": {
                    qid: ans.model_dump()
                    for qid, ans in merged_parent_answers.items()
                },
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    md_lines = [
        f"# Scorecard — {scorecard.subject} {scorecard.year}"
        + (f" ({scorecard.set_label})" if scorecard.set_label else ""),
        "",
        f"**{scorecard.total_points_earned:.1f} / {scorecard.total_points_possible:.1f}  ({scorecard.percentage:.1f}%)**",
        "",
    ]
    for qs in scorecard.questions:
        md_lines.append(f"## Question {qs.question_id} — {qs.points_earned:.1f} / {qs.points_possible:.1f}")
        for ps in qs.point_scores:
            check = "✅" if ps.awarded else "❌"
            md_lines.append(f"- {check} **{ps.point_id}** ({ps.points_earned:.1f} pt): {ps.rationale}")
        md_lines.append("")
    base.with_suffix(".md").write_text("\n".join(md_lines), encoding="utf-8")

    results.append((slug, subject, scorecard, html_path))
    print(f"  {scorecard.total_points_earned:.1f}/{scorecard.total_points_possible:.1f} "
          f"({scorecard.percentage:.1f}%) in {time.perf_counter() - _t0:.1f}s  ->  {html_path.name}")

fresh = len(results) - len(cached)
print(f"\nDone: {fresh} newly graded, {len(cached)} from cache, {len(failures)} failed.")

## Summary

A text summary of every folder graded, with its score and report file. The HTML reports are written to `out/` (open them in a browser) — they are intentionally **not** rendered inline here, because embedding the base64 page images bloats this notebook file.

In [ ]:
# Text-only summary. The HTML reports are written to out/ by the cell above;
# they are NOT rendered inline here, because embedding the base64 page images
# makes this .ipynb file huge. Open the .html files in out/ in a browser.
print(f"Graded {len(results)} exam(s); {len(failures)} failed.\n")

if results:
    print(f"{'Subject':<36}{'Score':>10}{'Pct':>8}   Report file")
    print("-" * 78)
    for slug, subject, sc, html_path in results:
        score = f"{sc.total_points_earned:.1f}/{sc.total_points_possible:.1f}"
        print(f"{subject:<36}{score:>10}{sc.percentage:7.1f}%   {html_path.name}")

if failures:
    print("\nFailed:")
    for slug, subject, err in failures:
        print(f"  - {subject} ({slug}): {err}")

print(f"\nAll reports written to: {CONFIG['output_dir'].resolve()}")